# Day 15 — Hypothesis Testing
> Understanding the logic of statistical hypothesis testing.

## What is Hypothesis Testing?

Hypothesis testing is a formal procedure to decide whether sample data provides enough evidence to reject a default assumption (H₀).

| Term | Definition |
|------|------------|
| **H₀** (Null Hypothesis) | The default assumption — no effect, no difference |
| **H₁** (Alternative Hypothesis) | What we want to prove |
| **α** (Significance Level) | Max acceptable probability of a Type I error (usually 0.05) |
| **Type I Error** | Rejecting H₀ when it is actually true (false positive) |
| **Type II Error** | Failing to reject H₀ when H₁ is true (false negative) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style='whitegrid')

# --- Simulate a coin-flip experiment ---
np.random.seed(42)
n_flips = 100
p_fair = 0.5        # H0: coin is fair
p_biased = 0.62     # true probability (unknown to the tester)

observed_heads = np.random.binomial(n_flips, p_biased)
print(f"Observed heads: {observed_heads} out of {n_flips}")
print(f"Observed proportion: {observed_heads/n_flips:.2f}")


## Decision Rule

We reject H₀ if the probability of observing our data (or more extreme) **under H₀** is less than α.

In [ ]:
# Binomial test: is this coin fair?
result = stats.binomtest(observed_heads, n=n_flips, p=p_fair, alternative='two-sided')
print(f"p-value: {result.pvalue:.4f}")
print(f"Decision: {'Reject H0 — coin is biased' if result.pvalue < 0.05 else 'Fail to reject H0'}")


## Visualizing the Sampling Distribution

In [ ]:
# Under H0, how likely is our observation?
x = np.arange(0, n_flips + 1)
pmf = stats.binom.pmf(x, n_flips, p_fair)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, pmf, alpha=0.5, color='steelblue', label='H₀ distribution (p=0.5)')
ax.axvline(observed_heads, color='red', lw=2, linestyle='--',
           label=f'Observed: {observed_heads}')
ax.axvline(n_flips - observed_heads, color='red', lw=2, linestyle='--')

# Shade rejection regions
crit_low = stats.binom.ppf(0.025, n_flips, p_fair)
crit_high = stats.binom.ppf(0.975, n_flips, p_fair)
ax.fill_between(x, pmf, where=(x <= crit_low) | (x >= crit_high),
                alpha=0.4, color='red', label='Rejection region (α=0.05)')

ax.set_xlabel('Number of Heads')
ax.set_ylabel('Probability')
ax.set_title('Sampling Distribution Under H₀')
ax.legend()
plt.tight_layout()
plt.savefig('../results/01_hypothesis_testing.png', dpi=150)
plt.show()
print(f"Critical region: ≤{int(crit_low)} or ≥{int(crit_high)}")


## Key Takeaways

- We never *prove* H₀ true — we only reject it or fail to reject it.
- A small p-value means the data is unlikely **if H₀ were true**.
- Setting α = 0.05 means we accept a 5% chance of a false positive.